In [ ]:
"""Figure 5E–J: Tracheole terminal-end spacing and Strahler order.

Input: CATMAID node table for 117 tracheoles, with skeleton_id, node_id,
parent_id, and x_nm/y_nm/z_nm coordinates. Terminal ends exclude roots.
Distances are in µm. Strahler-5 (n=1) is plotted but excluded from tests.
Statistical CIs use 5,000 bootstrap resamples; pairwise P values use BH FDR.
"""

from pathlib import Path
from collections import defaultdict, deque
from itertools import combinations

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist
from scipy.stats import gaussian_kde

folder = Path.home() / "Downloads"
node_csv = folder / "catmaid_nodes.csv"
out_dir = folder / "fig5"
out_dir.mkdir(parents=True, exist_ok=True)

n_boot, boot_seed, plot_seed = 5000, 20260909, 20260907
colors = {1: "#D62728", 2: "#FF7F0E", 3: "#2CA02C", 4: "#1F77B4", 5: "#9467BD"}


def load_nodes(path):
    nodes = pd.read_csv(path, low_memory=False)
    fields = ["skeleton_id", "node_id", "parent_id", "x_nm", "y_nm", "z_nm"]
    missing = set(fields) - set(nodes.columns)
    if missing:
        raise ValueError(f"Missing node columns: {sorted(missing)}")
    nodes = nodes[fields].copy()
    for name in ["skeleton_id", "node_id"]:
        nodes[name] = pd.to_numeric(nodes[name], errors="raise").astype(np.int64)
    nodes["parent_id"] = pd.to_numeric(nodes["parent_id"], errors="coerce").astype("Int64")
    for name in ["x_nm", "y_nm", "z_nm"]:
        nodes[name] = pd.to_numeric(nodes[name], errors="raise")
    if nodes.skeleton_id.nunique() != 117:
        raise ValueError("Expected 117 tracheoles")
    return nodes


def terminal_ends(nodes):
    parts = []
    for sid, group in nodes.groupby("skeleton_id", sort=True):
        ids = set(group.node_id.tolist())
        valid_parent = group.parent_id.notna() & group.parent_id.isin(ids)
        parents = set(group.loc[valid_parent, "parent_id"].astype(np.int64).tolist())
        roots = ~valid_parent
        ends = group.loc[~group.node_id.isin(parents) & ~roots].copy()
        parts.append(ends)
    ends = pd.concat(parts, ignore_index=True)
    for c in "xyz":
        ends[f"{c}_um"] = ends[f"{c}_nm"] / 1000.0
    if len(ends) != 1376:
        raise ValueError(f"Expected 1,376 terminal ends, found {len(ends)}")
    return ends


def strahler_by_skeleton(nodes):
    out = {}
    for sid, group in nodes.groupby("skeleton_id", sort=False):
        ids = group.node_id.astype(np.int64).tolist()
        id_set = set(ids)
        parent, children = {}, defaultdict(list)
        for row in group.itertuples(index=False):
            nid, pid = int(row.node_id), row.parent_id
            if pd.notna(pid) and int(pid) in id_set:
                pid = int(pid)
                parent[nid] = pid
                children[pid].append(nid)
            else:
                parent[nid] = None
        pending = {nid: len(children.get(nid, [])) for nid in ids}
        queue = deque(nid for nid in ids if pending[nid] == 0)
        order = {}
        while queue:
            nid = queue.popleft()
            child_orders = [order[c] for c in children.get(nid, [])]
            if not child_orders:
                order[nid] = 1
            else:
                highest = max(child_orders)
                order[nid] = highest + 1 if child_orders.count(highest) >= 2 else highest
            pid = parent.get(nid)
            if pid is not None:
                pending[pid] -= 1
                if pending[pid] == 0:
                    queue.append(pid)
        roots = [nid for nid in ids if parent[nid] is None]
        out[int(sid)] = int(max((order[r] for r in roots if r in order), default=max(order.values())))
    return out


def nearest_within(ends):
    parts = []
    for sid, group in ends.groupby("skeleton_id", sort=True):
        if len(group) < 2:
            continue
        dist = cdist(group[["x_um", "y_um", "z_um"]].to_numpy(np.float64),
                     group[["x_um", "y_um", "z_um"]].to_numpy(np.float64))
        np.fill_diagonal(dist, np.inf)
        nearest = dist[np.arange(len(group)), np.argmin(dist, axis=1)]
        part = group.copy()
        part["nearest_um"] = nearest
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def nearest_cross(ends):
    coords = ends[["x_um", "y_um", "z_um"]].to_numpy(np.float64)
    sids = ends.skeleton_id.to_numpy(np.int64)
    dist = cdist(coords, coords)
    np.fill_diagonal(dist, np.inf)
    dist[sids[:, None] == sids[None, :]] = np.inf
    result = ends.copy()
    result["nearest_um"] = dist[np.arange(len(dist)), np.argmin(dist, axis=1)]
    return result


def per_tracheole(ends):
    return ends.groupby(["skeleton_id", "strahler"], as_index=False).agg(mean_um=("nearest_um", "mean"))


def bootstrap_rho(x, y, n=n_boot, seed=boot_seed):
    x, y = np.asarray(x, np.float64), np.asarray(y, np.float64)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    if len(x) < 4:
        return np.nan, np.nan
    rng, boot = np.random.default_rng(seed), []
    for _ in range(n):
        idx = rng.integers(0, len(x), size=len(x))
        xb, yb = x[idx], y[idx]
        if np.unique(xb).size < 2 or np.unique(yb).size < 2:
            continue
        rho, _ = stats.spearmanr(xb, yb)
        if np.isfinite(rho):
            boot.append(float(rho))
    if len(boot) < 100:
        return np.nan, np.nan
    return tuple(np.quantile(np.asarray(boot), [0.025, 0.975]))


def bh_fdr(pvalues):
    pvalues = np.asarray(pvalues, np.float64)
    result = np.full(len(pvalues), np.nan)
    valid = np.isfinite(pvalues)
    p = pvalues[valid]
    if len(p) == 0:
        return result
    ranks = np.argsort(p)
    adjusted = p[ranks] * len(p) / np.arange(1, len(p) + 1, dtype=np.float64)
    adjusted = np.clip(np.minimum.accumulate(adjusted[::-1])[::-1], 0, 1)
    sorted_back = np.empty_like(adjusted)
    sorted_back[ranks] = adjusted
    result[valid] = sorted_back
    return result


def test_strahler(summary, pairwise=False):
    d = summary[["skeleton_id", "strahler", "mean_um"]].dropna().copy()
    orders = [order for order in sorted(d.strahler.unique()) if (d.strahler == order).sum() >= 2]
    d = d[d.strahler.isin(orders)]
    groups = [d.loc[d.strahler == order, "mean_um"].to_numpy(np.float64) for order in orders]
    H, p = stats.kruskal(*groups)
    rho, rho_p = stats.spearmanr(d.strahler, d.mean_um)
    ci_low, ci_high = bootstrap_rho(d.strahler.to_numpy(np.float64), d.mean_um.to_numpy(np.float64))
    report = {"orders": orders, "n": len(d), "H": float(H), "df": len(orders) - 1,
              "P": float(p), "rho": float(rho), "rho_P": float(rho_p),
              "rho_CI_low": float(ci_low), "rho_CI_high": float(ci_high)}
    pairs = []
    if pairwise:
        for a, b in combinations(orders, 2):
            va = d.loc[d.strahler == a, "mean_um"].to_numpy(np.float64)
            vb = d.loc[d.strahler == b, "mean_um"].to_numpy(np.float64)
            U, p_raw = stats.mannwhitneyu(va, vb, alternative="two-sided", method="auto")
            pairs.append({"order_1": a, "order_2": b, "U": U, "P_raw": p_raw,
                          "rank_biserial": 2.0 * U / (len(va) * len(vb)) - 1.0})
    pairs = pd.DataFrame(pairs)
    if len(pairs):
        pairs["P_BH_FDR"] = bh_fdr(pairs.P_raw.to_numpy(np.float64))
    return report, pairs


def save_fig(fig, panel):
    fig.savefig(out_dir / f"fig5{panel}.svg", bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


def open_axes(ax, large=False):
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    if large:
        for side in ["left", "bottom"]:
            ax.spines[side].set_linewidth(1.1)
        ax.grid(False)


def plot_ecdf(values, panel, xlabel):
    plt.rcParams.update({"font.family": "Arial", "font.size": 8.5, "axes.labelsize": 8.5,
                         "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.linewidth": 0.8,
                         "xtick.major.width": 0.8, "ytick.major.width": 0.8,
                         "xtick.major.size": 3, "ytick.major.size": 3, "svg.fonttype": "none"})
    vals = np.asarray(values, np.float64)
    vals = vals[np.isfinite(vals)]
    median = float(np.median(vals))
    x = np.sort(vals)
    y = np.arange(1, len(x) + 1) / len(x)
    fig, ax = plt.subplots(figsize=(3.5, 1.87))
    ax.plot(x, y, color="black", linewidth=1.6)
    ax.axvline(median, color="0.35", linestyle="--", linewidth=0.9)
    if panel == "e":
        ax.text(median, 0.08, f"median\n{median:.2f} µm", rotation=90,
                ha="right", va="bottom", fontsize=7.5, color="0.25")
        ax.set_xlim(0, max(5.0, np.ceil(float(np.max(vals)) / 5.0) * 5.0))
    else:
        ax.text(median, 0.955, f"{median:.2f} µm", ha="center", va="top",
                fontsize=7.5, color="0.25")
        ax.set_xlim(0, 65)
    ax.set(xlabel=xlabel, ylabel="Cumulative fraction", ylim=(0, 1.01))
    open_axes(ax)
    fig.tight_layout()
    save_fig(fig, panel)
    return median


def plot_ranked(summary, panel):
    ranked = summary.sort_values("mean_um").reset_index(drop=True)
    rank = np.arange(1, len(ranked) + 1)
    overall_mean = float(ranked.mean_um.mean())
    bar_colors = [colors.get(int(order), "0.45") for order in ranked.strahler]
    if panel == "f":
        plt.rcParams.update({"font.family": "Arial", "font.size": 9, "axes.labelsize": 10,
                             "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5,
                             "axes.linewidth": 0.8, "svg.fonttype": "none"})
        fig, ax = plt.subplots(figsize=(7.2, 3.6))
        ax.bar(rank, ranked.mean_um, width=0.92, color=bar_colors, edgecolor="none", linewidth=0, zorder=2)
        ax.axhline(overall_mean, color="0.40", linestyle="--", linewidth=1.0, zorder=3)
        ymax = float(ranked.mean_um.max())
        ax.text(1.5, overall_mean + 0.025 * ymax, f"Mean = {overall_mean:.2f} µm",
                ha="left", va="bottom", fontsize=8.5, color="0.30")
        ax.set(xlabel="Tracheole rank", ylabel="Mean within-tracheole\nnearest-end distance (µm)",
               xlim=(0.3, len(ranked) + 0.7), ylim=(0, ymax * 1.08))
        ax.set_xticks([1, len(ranked)])
        ax.set_xticklabels(["1", str(len(ranked))])
        legend = [Patch(facecolor=colors.get(int(o), "#666666"), edgecolor="none", label=f"Strahler {o}")
                  for o in sorted(ranked.strahler.unique())]
        ax.legend(handles=legend, frameon=False, loc="upper left", ncol=2,
                  columnspacing=1.5, handlelength=1.5, handleheight=0.9)
        open_axes(ax)
        fig.tight_layout()
    else:
        plt.rcParams.update({"font.family": "Arial", "font.size": 13, "axes.labelsize": 14,
                             "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11.5,
                             "axes.linewidth": 1.1, "xtick.direction": "out", "ytick.direction": "out",
                             "svg.fonttype": "none", "pdf.fonttype": 42})
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.bar(rank, ranked.mean_um, width=0.88, color=bar_colors, edgecolor="none",
               linewidth=0, zorder=2, rasterized=True)
        ax.axhline(overall_mean, color="0.35", linestyle="--", linewidth=1.4, zorder=4)
        ymin, ymax = float(ranked.mean_um.min()), float(ranked.mean_um.max())
        ax.set(xlim=(0.2, len(ranked) + 0.8), ylim=(0, ymax + 0.1 * ((ymax - ymin) if ymax > ymin else 1.0)),
               xlabel="Tracheole rank", ylabel="Mean nearest-end distance (µm)")
        ticks = np.unique(np.round(np.linspace(1, len(ranked), 6)).astype(int))
        ax.set_xticks(ticks)
        legend = [Patch(facecolor=colors.get(int(o), "0.45"), edgecolor="none", label=f"Strahler {o} ")
                  for o in sorted(ranked.strahler.unique())]
        legend.append(Line2D([0], [0], color="0.35", linestyle="--", linewidth=1.4,
                             label=f"Overall mean ({overall_mean:.2f} µm)"))
        ax.legend(handles=legend, frameon=False, loc="upper left", ncol=2,
                  fontsize=11.5, handlelength=1.3, handletextpad=0.5, columnspacing=1.4,
                  borderaxespad=0.3)
        open_axes(ax, large=True)
        fig.tight_layout(pad=1.5)
    save_fig(fig, panel)


def format_p(p):
    return "P < 0.001" if p < 0.001 else f"P = {p:.3f}"


def plot_strahler(summary, stats_result, panel, ylabel):
    plt.rcParams.update({"font.family": "Arial", "font.size": 13, "axes.labelsize": 14,
                         "xtick.labelsize": 12, "ytick.labelsize": 12, "axes.linewidth": 1.1,
                         "xtick.direction": "out", "ytick.direction": "out",
                         "svg.fonttype": "none", "pdf.fonttype": 42})
    d = summary.reset_index(drop=True)
    orders = sorted(d.strahler.unique())
    vals_all = d.mean_um.to_numpy(float)
    span = float(vals_all.max() - vals_all.min())
    if span <= 0:
        span = 1.0
    fig, ax = plt.subplots(figsize=(8, 5))
    rng = np.random.default_rng(plot_seed)
    for xpos, order in enumerate(orders, start=1):
        values = d.loc[d.strahler == order, "mean_um"].to_numpy(float)
        color = colors.get(int(order), "0.45")
        if len(values) >= 2 and np.std(values) > 0:
            kde = gaussian_kde(values)
            local_span = values.max() - values.min()
            if local_span <= 0:
                local_span = span
            grid = np.linspace(values.min() - 0.06 * local_span,
                               values.max() + 0.06 * local_span, 300)
            density = kde(grid)
            density = density / density.max() * 0.28
            ax.fill_betweenx(grid, xpos - density, xpos, facecolor=color, alpha=0.20, linewidth=0, zorder=1)
            ax.plot(xpos - density, grid, color=color, linewidth=1.25, alpha=0.90, zorder=2)
        jitter = rng.uniform(-0.075, 0.075, size=len(values))
        ax.scatter(xpos + 0.11 + jitter, values, s=34, color=color, alpha=0.78,
                   edgecolors="none", zorder=3, rasterized=False)
        q1, med, q3 = np.quantile(values, [0.25, 0.5, 0.75])
        ax.plot([xpos + 0.27] * 2, [q1, q3], color="#202020", linewidth=2.0,
                solid_capstyle="round", zorder=5)
        ax.scatter(xpos + 0.27, med, s=52, facecolor="#111111", edgecolor="white",
                   linewidth=0.8, zorder=6)
    ax.set(xlim=(0.45, len(orders) + 0.55), ylim=(0, float(vals_all.max()) + 0.14 * span),
           xlabel="Strahler order", ylabel=ylabel)
    ax.set_xticks(range(1, len(orders) + 1))
    ax.set_xticklabels([f"{order}\nn={(d.strahler == order).sum()}" for order in orders], fontsize=12.5)
    stat_text = f"Kruskal–Wallis {format_p(stats_result['P'])}\nSpearman ρ = {stats_result['rho']:.2f}, {format_p(stats_result['rho_P'])}"
    ax.text(0.98, 0.97, stat_text, transform=ax.transAxes, ha="right", va="top",
            fontsize=10.5, color="#222222", linespacing=1.35, zorder=10)
    open_axes(ax, large=True)
    fig.tight_layout(pad=1.5)
    save_fig(fig, panel)


nodes = load_nodes(node_csv)
ends = terminal_ends(nodes)
ends["strahler"] = ends.skeleton_id.map(strahler_by_skeleton(nodes)).astype(int)

within = nearest_within(ends)
cross = nearest_cross(ends)
within_summary = per_tracheole(within)
cross_summary = per_tracheole(cross)

within_stats, within_pairs = test_strahler(within_summary, pairwise=True)
cross_stats, _ = test_strahler(cross_summary)

within_median = plot_ecdf(within.nearest_um, "e", "Nearest same-tracheole terminal-end distance (µm)")
plot_ranked(within_summary, "f")
plot_strahler(within_summary, within_stats, "g", "Mean within-tracheole\nnearest-end distance (µm)")
cross_median = plot_ecdf(cross.nearest_um, "h", "Nearest cross-tracheole end distance (µm)")
plot_ranked(cross_summary, "i")
plot_strahler(cross_summary, cross_stats, "j", "Mean cross-tracheole\nnearest-end distance (µm)")

report = pd.DataFrame([{"panel": "5G", **within_stats}, {"panel": "5J", **cross_stats}])
print(f"Figure 5E median: {within_median:.2f} µm; Figure 5H median: {cross_median:.2f} µm")
print(report.to_string(index=False))
print("Figure 5G pairwise tests:")
print(within_pairs.to_string(index=False))